# ABCD NDA Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
# import globus_sdk


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


In [4]:
# path setting
main_dir = os.path.abspath('../../..')


## Downloading data from NDA s3 bucket storage

---

Note: this is no longer useful ABCD downloads moved to Globus.

In [ ]:
# downloaded by:
# cd /mountpoint/data/NDA
# downloadcmd -dp 1240448 -u 'sinamansourlakouraj' -d ABCD5p0ImgManifest -wt 8


## Downloading data from ABCD Globus endpoint

---

Note: after close examination, NDA (release 5.1) data does not include freesurfer outputs. However, fortunately, release 6.0 provides freesurfer outputs, but unfortunately, these outputs are provided with an entirely different download procedure. Hence, scripts bellow are used:

Long story short:

The approach that does what I want is achieved by:

- First install and configure globus connect personal (`./globusconnectpersonal -setup`)
- Configure authentication via Lasso
- Install globus cli and login (`globus login`)
- Initiate the following transfer command

```bash
globus transfer \
    "41494652-d3e1-498d-b97a-5ecb65b323ed:/dairc/derivatives/freesurfer/" \
    "<destination_endpoint>:/mountpoint/data/Globus/data/ABCD/freesurfer/" \
    --recursive --label "ABCD freesurfer selective derivative transfer" \
    --include "lh.white" --include "rh.white" \
    --include "lh.pial" --include "rh.pial" \
    --include "lh.thickness" --include "rh.thickness" \
    --include "lh.orig.nofix" --include "rh.orig.nofix" \
    --include "lh.sphere.reg" --include "rh.sphere.reg" \
    --exclude "*" --verify-checksum --dry-run

# Remove the dry run flag for execution
```


In [28]:
import subprocess

globus_exe = "/mountpoint/code/environments/pymc_env/bin/globus"
endpoint_id_path = "41494652-d3e1-498d-b97a-5ecb65b323ed:/dairc/derivatives/freesurfer/"
result = subprocess.run([globus_exe, "ls", endpoint_id_path], capture_output=True, text=True)
subjects = [x[:-1] for x in result.stdout.split('\n') if x[:4] == 'sub-']


## Extracting data

---

In [4]:
abcd_dir = '/mountpoint/data/Globus/data/ABCD/freesurfer'
abcd_subjects = [x.split('/')[-1] for x in list_dirs(abcd_dir)]
len(abcd_subjects)


30318

In [5]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]
abcd_valid_subjects = [
    subject for subject in abcd_subjects
    if all([
        file_exists(f'/mountpoint/data/Globus/data/ABCD/freesurfer/{subject}/surf/{item}','')
        for item in items
    ])
]
len(abcd_valid_subjects)


30290

In [7]:
# ignore warning
nib.imageglobals.logger.setLevel(40)


In [ ]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(abcd_valid_subjects)):
    sub_dir = f"{idx:02d}"[-2:]
    freesurfer_directory = f"/mountpoint/data/Globus/data/ABCD/freesurfer/{subject}/"
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
    np.save(
        ensure_dir(f"/mountpoint/data/normative/fs_LR_32k/ABCD/{sub_dir}/{subject}.thickness.fslr.npy"),
        transformed_fslr_thickness.astype(np.float32)
    )


  0%|          | 0/30290 [00:00<?, ?it/s]

ABCD demography information

In [ ]:
# empty dict to hold information
abcd_valid_subjects_dict = {
    subject: {
        "participant_id": subject.split("_")[0],
        "session_id": subject.split("_")[1],
        "subject_index": idx,
    }
    for idx, subject in enumerate(tqdm(abcd_valid_subjects))
}

len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:3]


In [ ]:
participant_session_list_dict = {}

for key in abcd_valid_subjects_dict:
    session_list = participant_session_list_dict.get(abcd_valid_subjects_dict[key]["participant_id"], [])
    session_list.append(key)
    participant_session_list_dict[abcd_valid_subjects_dict[key]["participant_id"]] = session_list

len(participant_session_list_dict), list(participant_session_list_dict.items())[:3]


In [ ]:
%%time

ab_g_dyn = pd.read_parquet("/mountpoint/data/Globus/data/ABCD/tabular/dairc/rawdata/phenotype/ab_g_dyn.parquet")
ab_g_stc = pd.read_parquet("/mountpoint/data/Globus/data/ABCD/tabular/dairc/rawdata/phenotype/ab_g_stc.parquet")
ab_p_screen = pd.read_parquet("/mountpoint/data/Globus/data/ABCD/tabular/dairc/rawdata/phenotype/ab_p_screen.parquet")

sex_reformater = {"1": "M", "2": "F"}  # https://docs.abcdstudy.org/latest/documentation/non_imaging/ab.html#ab_g_stc

for key in abcd_valid_subjects_dict:
    abcd_valid_subjects_dict[key]["unique_id"] = key


for idx, row in ab_g_dyn.iterrows():
    key = f"{row['participant_id']}_{row['session_id']}"
    if key in abcd_valid_subjects_dict:
        abcd_valid_subjects_dict[key]["age"] = row["ab_g_dyn__visit_age"]
        abcd_valid_subjects_dict[key]["site"] = f'site:{row["ab_g_dyn__design_site"]}_scanner:{row["ab_g_dyn__design_mr__serial"]}' ## there are several potential site identifiers (site vs. scanner model/serial)

for idx, row in ab_g_stc.iterrows():
    for key in participant_session_list_dict.get(row['participant_id'], []):
        abcd_valid_subjects_dict[key]["sex"] = sex_reformater[row["ab_g_stc__cohort_sex"]]
        abcd_valid_subjects_dict[key]["family_id"] = row["ab_g_stc__design_id__fam"]
        

# diagnoses for screening/exclusion
for idx, row in ab_p_screen.iterrows():
    for key in participant_session_list_dict.get(row['participant_id'], []):
        abcd_valid_subjects_dict[key]["diagnosis"] = any([
            (row["ab_p_screen__med_002"] == '1'),  # cerebral_palsy_diagnoses
            (row["ab_p_screen__med_003"] == '1'),  # brain_tumor_diagnoses
            (row["ab_p_screen__med_004"] == '1'),  # stroke_diagnoses
            (row["ab_p_screen__med_005"] == '1'),  # brain_aneurysm_diagnoses
            (row["ab_p_screen__med_006"] == '1'),  # brain_hemorrhage_diagnoses
            (row["ab_p_screen__med_007"] == '1'),  # subdural_hematoma_diagnoses
            (row["ab_p_screen__med_008"] == '1'),  # epilepsy_or_seizures_diagnoses
            (row["ab_p_screen__med_011"] == '1'),  # adhd,_depression,_Bipolar_disorder,_anxiety,_phobias,_diagnoses
            (row["ab_p_screen__med_012"] == '1'),  # schizophrenia_diagnoses
            (row["ab_p_screen__med_013"] == '1'),  # autism_spectrum_disorder_diagnoses
            (row["ab_p_screen__med_015"] == '1'),  # alcohol_or_substance_use_disorder_diagnoses
            (row["ab_p_screen__med_016"] == '1'),  # intellectual_disability_diagnoses
            (row["ab_p_screen__med_017"] == '1')  # psychological_or_psychiatric_diagnoses
        ])

for idx, row in mr_y_qc__incl.iterrows():
    key = f"{row['participant_id']}_{row['session_id']}"
    if key in abcd_valid_subjects_dict:
        abcd_valid_subjects_dict[key]["recommended"] = (row["mr_y_qc__incl__smri__t1_indicator"] == '1')

len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:1]


In [120]:
len({
    abcd_valid_subjects_dict[x]['participant_id']
    for x in abcd_valid_subjects_dict
    if (
        (not abcd_valid_subjects_dict[x].get('diagnosis', False)) and
        (abcd_valid_subjects_dict[x].get('recommended', False))
    )
})

9635

In [121]:
len([
    x for x in abcd_valid_subjects_dict
    if (
        (not abcd_valid_subjects_dict[x].get('diagnosis', False)) and
        (abcd_valid_subjects_dict[x].get('recommended', False))
    )
])

24339

In [ ]:
for key in tqdm(abcd_valid_subjects_dict):
    abcd_valid_subjects_dict[key]["euler_no"] = float(snm.utils.nitools.compute_total_euler_number(
        f"/mountpoint/data/Globus/data/ABCD/freesurfer/{key}/"
    ))
    
len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:3]


In [ ]:
for key in tqdm(abcd_valid_subjects_dict):
    idx = abcd_valid_subjects_dict[key]["subject_index"]
    subject = abcd_valid_subjects_dict[key]["unique_id"]
    abcd_valid_subjects_dict[key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/ABCD/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()

    
len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:1]


In [ ]:
for key in tqdm(abcd_valid_subjects_dict):
    abcd_valid_subjects_dict[key]["validity_check"] = (
        abcd_valid_subjects_dict[key].get('recommended', False) and
        (not abcd_valid_subjects_dict[key].get('diagnosis', False))
    )
    
len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:1]


In [153]:
import joblib

joblib.dump(abcd_valid_subjects_dict, ensure_dir("/mountpoint/data/normative/datasets/ABCD/subjects.joblib"))


['/mountpoint/data/normative/datasets/ABCD/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
abcd_valid_subjects_dict = joblib.load(
    "/mountpoint/data/normative/datasets/ABCD/subjects.joblib"
)

len(abcd_valid_subjects_dict), list(abcd_valid_subjects_dict.items())[:1]


In [ ]:
abcd_df = pd.DataFrame({
    'age': [abcd_valid_subjects_dict[key]["age"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'thickness': [abcd_valid_subjects_dict[key]["thickness"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'sex': [abcd_valid_subjects_dict[key]["sex"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'site': [abcd_valid_subjects_dict[key]["site"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [abcd_valid_subjects_dict[key]["participant_id"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'euler_no': [abcd_valid_subjects_dict[key]["euler_no"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [abcd_valid_subjects_dict[key]["unique_id"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
    'subject_index': [abcd_valid_subjects_dict[key]["subject_index"] for key in abcd_valid_subjects_dict if abcd_valid_subjects_dict[key]["validity_check"]],
})
dataset_name = 'ABCD'
abcd_df['dataset'] = dataset_name
abcd_df.head(), abcd_df.shape


In [14]:
# randomly select only one timepoint per subject (cross-sectional sample)
abcd_df_subset = abcd_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# Keep only sites with at least 15 subjects
subjects_per_site = abcd_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

abcd_df_subset[abcd_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset_name}/demography.parquet')
)

abcd_df_subset[abcd_df_subset["site"].isin(valid_sites)].shape


(9612, 9)

In [ ]:
np.save(
    ensure_dir("/mountpoint/data/normative/datasets/ABCD/subjects.npy"),
    np.array(abcd_valid_subjects)
)
